# 01 - EDA BMKG + BPS

Tujuan:
- Distribusi data per kelurahan
- Korelasi antar variabel (climate, infra, sosek)
- Missing value analysis dan strategi imputasi awal
- Outlier detection sebelum model training

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", 200)

In [ ]:
dataRoot = Path("../data")
rawDir = dataRoot / "raw"

dataFiles = {
    "bmkg": rawDir / "bmkg.csv",
    "bps": rawDir / "bps.csv",
    "osm": rawDir / "osm.csv",
    "bnpb": rawDir / "bnpb.csv"
}

dataFrames = {}

for name, path in dataFiles.items():
    if path.exists():
        dataFrames[name] = pd.read_csv(path)
    else:
        print(f"Missing: {path}")

print("Loaded:", list(dataFrames.keys()))

In [ ]:
for name, df in dataFrames.items():
    print(name, df.shape)
    display(df.head(3))

In [ ]:
def missingReport(df):
    missRate = df.isna().mean().sort_values(ascending=False)
    return missRate.to_frame("missingRate")

for name, df in dataFrames.items():
    print(name)
    display(missingReport(df).head(20))

In [ ]:
kelurahanCol = "kelurahan"
valueCol = None  # set a numeric column, ex: "rainfall"

for name, df in dataFrames.items():
    if kelurahanCol not in df.columns:
        continue
    topCounts = df[kelurahanCol].value_counts().head(20)
    ax = topCounts.plot(kind="bar", title=f"{name} - top kelurahan count")
    ax.set_xlabel("kelurahan")
    ax.set_ylabel("count")
    plt.show()

    if valueCol and valueCol in df.columns:
        df[valueCol].hist(bins=30)
        plt.title(f"{name} - {valueCol} distribution")
        plt.show()

In [ ]:
for name, df in dataFrames.items():
    numCols = df.select_dtypes(include="number").columns
    if len(numCols) < 2:
        continue
    corr = df[numCols].corr()
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, cmap="coolwarm", center=0)
    plt.title(f"{name} - correlation")
    plt.show()

In [ ]:
def outlierRate(df, z=3.0):
    numCols = df.select_dtypes(include="number").columns
    if len(numCols) == 0:
        return pd.Series(dtype=float)
    mean = df[numCols].mean()
    std = df[numCols].std(ddof=0).replace(0, np.nan)
    zScores = (df[numCols] - mean) / std
    rate = (zScores.abs() > z).mean().sort_values(ascending=False)
    return rate

for name, df in dataFrames.items():
    rate = outlierRate(df)
    if rate.empty:
        continue
    print(name)
    display(rate.head(20))

Imputation plan (fill in):
- Numeric: median per kelurahan or city-wide
- Categorical: mode
- Add isImputed flags for key features